# KRAS — a canonical rescue

Two adjacent SNVs in KRAS exon 2:

- `mut1` = `KRAS:12:25227343:G:T`
- `mut2` = `KRAS:12:25227344:A:T`

The canonical donor of exon 2 is at chr12:25,227,234 (negative strand). We expect `mut1` alone to disrupt that donor and `mut2` alone to leave it intact. The interesting question is whether the joint event recovers canonical splicing — and if so, why.

In [ ]:
from oncosplice import OncospliceEngine

eng = OncospliceEngine(splicing_engine='spliceai_pytorch')
pair = eng.analyze_pair(
    'KRAS:12:25227343:G:T',
    'KRAS:12:25227344:A:T',
    protein=False,
)
print('pair_classification:', pair.pair_classification)
print('max |residual|     :', pair.site_residuals.residual.abs().max())

## Per-site mechanism

Pull the site(s) the classifier flagged. We expect a single rescue site at the canonical donor.

In [ ]:
sr = pair.site_residuals
epi = sr[sr.classification != 'non-epistatic']
epi[['position','site_type','annotated','ref','mut1','mut2','event','residual','classification']]

**Reading the row**: at D 25,227,234 the WT probability is ~0.97 (a strong canonical donor). `mut1` alone destroys it (drops to 0.13). `mut2` alone barely touches it (0.98 ≈ WT). The joint event restores the donor to ~0.97 — `mut2`'s presence rescues splicing despite `mut1`'s disruptive effect.

Mechanistically this means `mut2` either re-creates a splice signal that `mut1` destroyed, or compensates for the change in upstream/downstream context that `mut1` introduced.

## Case-study figure

The bar plot shows splice probability at each annotated and active cryptic site, in WT / mut1 / mut2 / expected / joint contexts. Sites the classifier flagged as epistatic are labeled with their class + residual underneath.

In [ ]:
fig, _ = pair.plot_case_study(max_sites=8)
fig